In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id= dbutils.widgets.get("p_batch_id")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
races_schema = StructType([
    StructField('season', IntegerType()),
    StructField('round', IntegerType()),
    StructField('url', StringType()),
    StructField('raceName', StringType()),
    StructField('date', DateType()),
    StructField('circuitId', StringType())

])

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file= f"{landing_folder_path}/{v_batch_id}/races.csv"
table_name= f"{catalog_name}.{bronze_schema}.races"

In [0]:
races_df =( spark.read
           .format("csv")
           .option("header", True)
          # .option("inferSchema", True)
          .option('mode', 'FAILFAST')
           .schema(races_schema)
           .load(source_file))

In [0]:
races_final_df= add_ingestion_metadata(races_df)

In [0]:
display(races_final_df)

In [0]:
write_to_bronze (
    input_df = races_final_df,
    target_name = table_name,
    batch_id = v_batch_id
)

In [0]:
display(spark.read.table(table_name))